# AI Creative Studio

Create images with [Stable Diffusion v1.5](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5) in Google Colab.

1. Select **Runtime → Change runtime type → GPU**.
2. Run the cells in order. The first model download can take several minutes.
3. Enter a prompt and use the Gradio panel or the single-image example.

No Google Drive access is required. Generated images are AI outputs; review them before sharing. The model has its own [CreativeML OpenRAIL-M license](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5), separate from this repository's MIT code license.

## 1. Install Python libraries

Keep the PyTorch build supplied by the Colab GPU runtime. Do not replace it with a hard-coded CUDA wheel.

In [ ]:
%pip install -q "diffusers>=0.30,<1" "transformers>=4.44,<5" "accelerate>=0.30,<2" "gradio>=5,<6" "safetensors>=0.4" "pillow>=10"

## 2. Check the GPU and load the model

If no CUDA GPU is available, switch the Colab runtime to GPU before continuing.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select Runtime → Change runtime type → GPU, then rerun.")

MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
pipeline = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
).to("cuda")
pipeline.enable_attention_slicing()
print("Model ready on", torch.cuda.get_device_name(0))

## 3. Generate one image

Change the prompt or seed and rerun this cell. Use Colab's Files panel to download the image if you want to keep it.

In [ ]:
from IPython.display import display

prompt = "a cozy robot cafe, warm lighting, digital art"
negative_prompt = "blurry, low quality"
seed = 42
generator = torch.Generator(device="cuda").manual_seed(seed)

with torch.inference_mode():
    image = pipeline(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generator,
    ).images[0]

display(image)

## 4. Open the interactive generator

Prompt, negative prompt, inference steps, guidance scale, and seed are adjustable. No Google Drive mount is needed.

In [ ]:
import gradio as gr

def generate_image(prompt, negative_prompt, steps, guidance_scale, seed):
    if not prompt or not prompt.strip():
        raise gr.Error("Please enter a prompt before generating an image.")
    generator = torch.Generator(device="cuda").manual_seed(int(seed))
    with torch.inference_mode():
        return pipeline(
            prompt=prompt.strip(),
            negative_prompt=(negative_prompt or "").strip() or None,
            num_inference_steps=int(steps),
            guidance_scale=float(guidance_scale),
            generator=generator,
        ).images[0]

demo = gr.Interface(
    fn=generate_image,
    inputs=[
        gr.Textbox(label="Prompt", lines=3),
        gr.Textbox(label="Negative prompt (optional)", lines=2),
        gr.Slider(10, 50, value=30, step=1, label="Inference steps"),
        gr.Slider(1.0, 15.0, value=7.5, step=0.5, label="Guidance scale"),
        gr.Slider(0, 2147483647, value=42, step=1, label="Seed"),
    ],
    outputs=gr.Image(type="pil", label="Generated image"),
    title="AI Creative Studio",
    description="Create an image with Stable Diffusion v1.5 on a Colab GPU.",
    examples=[
        ["cyberpunk city with neon lights, rainy streets", "blurry, low quality", 30, 7.5, 42],
        ["a cozy robot cafe, warm lighting, digital art", "blurry, low quality", 30, 7.5, 123],
    ],
)
demo.launch()